# Function 5: 4D black-box maximisation

You’re tasked with optimising a four-variable black-box function that represents the yield of a chemical process in a factory. The function is typically unimodal, with a single peak where yield is maximised. Your goal is to find the optimal combination of chemical inputs that delivers the highest possible yield, using systematic exploration and optimisation methods.

## How this notebook works

1. **Data loading:** observations are read from `data.csv`; the incumbent is the best observed yield.
2. **Local geometry:** the observations nearest the incumbent, with their distances and changes in yield.
3. **Surrogates:** two ARD Matérn 5/2 Gaussian processes, one on raw `y` and one on `log1p(y)`, each validated by leave-one-out CV.
4. **Conditional local search:** x1 is held at the incumbent value and a dense grid over x2, x3 and x4 in the top of the domain is scored by posterior mean, EI, PI and UCB under the raw-`y` GP.
5. **Robustness checks:** the log-target GP is evaluated on the same grid, and compared against the observations nearest the upper corner.
6. **Recommendation:** the raw-`y` posterior-mean point is printed in submission format, with a check that the log-target GP agrees.

Everything is derived from `data.csv`: append new rows and re-run, with no code edits.

**Data-handling choices.**

- *Skewed target.* Yields span about 0.1 to 8,700, so a log1p model is fitted alongside the raw one.
- *Boundary optimum.* The best observed yield sits on the upper boundary of the domain, so local search bounds are clipped to `DOMAIN` and boundary hits are flagged.
- *Input format.* The recommended point is printed to 6 decimals, as the portal requires.

In [1]:
# =============================================================================
# IMPORTS, SETTINGS AND HELPERS
# =============================================================================

import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneOut

# GP fitting regularly reports optimiser convergence warnings; they do not
# change the fitted models, so they are suppressed.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

FUNCTION_ID = 5
DATA_PATH = "../data/data.csv"
SEED, GRID_RES = 42, 60
GP_LOO_RESTARTS, GP_FINAL_RESTARTS = 10, 25   # optimiser restarts: cross-validation / final fit
XI, KAPPA = 0.01, 1.5                         # EI exploration offset, UCB multiplier
DOMAIN = (0.0, 1.0)                           # admissible input range
# Local search width below the upper bound for each free input (x2, x3, x4);
# x1 is held fixed at the incumbent value.
LOCAL_FRACTIONS = (0.10, 0.15, 0.10)

def gp_model(dim, log_target=False, restarts=GP_FINAL_RESTARTS):
    """ARD Matérn-5/2 GP with a white-noise term (one length scale per input)."""
    kernel = (ConstantKernel(1.0, (1e-3, 1e3))
              * Matern(np.ones(dim), (1e-2, 1e3 if log_target else 1e2), nu=2.5)
              + WhiteKernel(1e-3, (1e-6, 1e1)))
    return GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                    n_restarts_optimizer=restarts, random_state=SEED)
    
def loo_gp(X, target, log_target=False):
    """Leave-one-out predictive means for the GP (refitted on each fold)."""
    mean = np.empty(len(target))
    for train, test in LeaveOneOut().split(X):
        model = gp_model(X.shape[1], log_target, GP_LOO_RESTARTS).fit(X[train], target[train])
        mean[test] = model.predict(X[test])
    return mean

def scores(actual, predicted):
    """RMSE, MAE and R² of predictions on the original yield scale."""
    return (np.sqrt(mean_squared_error(actual, predicted)),
            mean_absolute_error(actual, predicted), r2_score(actual, predicted))

def ard_table(model):
    """Input sensitivity as reciprocal fitted length scale (larger = more influential)."""
    scales = np.atleast_1d(model.kernel_.k1.k2.length_scale)
    return pd.DataFrame({'Feature': feature_cols, 'Length_Scale': scales,
                         'Sensitivity_1_over_L': 1 / scales}).sort_values('Sensitivity_1_over_L', ascending=False)

def posterior_search(model, candidates, log_target=False, batch=10_000):
    """GP posterior mean/std on the yield scale, evaluated in batches to limit memory.

    For a log1p-target GP the mean is back-transformed with expm1 and the std
    is the half-width of the back-transformed ±1σ interval.
    """
    means, stds = [], []
    for start in range(0, len(candidates), batch):
        mu, sigma = model.predict(candidates[start:start + batch], return_std=True)
        if log_target:
            sigma = (np.expm1(mu + sigma) - np.expm1(mu - sigma)) / 2
            mu = np.expm1(mu)
        means.append(mu); stds.append(sigma)
    return np.concatenate(means), np.concatenate(stds)


def format_point(x):
    """Format a point as the dash-separated six-decimal submission string."""
    return "-".join(f"{v:.6f}" for v in x)

In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()
feature_cols = [f"x{i}" for i in range(1, 9) if function_data[f"x{i}"].notna().any()]
X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()

df = function_data[[*feature_cols, 'y', 'round']].rename(columns={'round': 'Round'})
df['Stage'] = np.where(df['Round'].eq(0), 'Initial Data', 'Week ' + df['Round'].astype(str))

inc_idx = int(np.argmax(y))
incumbent_x, incumbent_y = X[inc_idx].copy(), float(y[inc_idx])

print(f'Function {FUNCTION_ID}: {len(y)} observations, {len(feature_cols)} dimensions; rounds 0–{df.Round.max()}')
print('Features:', feature_cols)
print('Incumbent:', dict(zip(feature_cols, incumbent_x)), 'yield:', incumbent_y)

Function 5: 32 observations, 4 dimensions; rounds 0–12
Features: ['x1', 'x2', 'x3', 'x4']
Incumbent: {'x1': np.float64(1.0), 'x2': np.float64(1.0), 'x3': np.float64(1.0), 'x4': np.float64(1.0)} yield: 8662.4825


In [3]:
# =============================================================================
# LOCAL GEOMETRY AROUND THE INCUMBENT
# =============================================================================
# The ten observations closest to the incumbent, with their distance and the
# change in yield relative to it.

geometry = df[['Round', *feature_cols, 'y']].copy()
geometry['Distance'] = np.linalg.norm(X - incumbent_x, axis=1)
geometry['Delta_y'] = y - incumbent_y

print(geometry.nsmallest(10, 'Distance').to_string(index=False, float_format=lambda v: f'{v:.6f}'))

 Round       x1       x2       x3       x4           y  Distance      Delta_y
     6 1.000000 1.000000 1.000000 1.000000 8662.482500  0.000000     0.000000
    10 1.000000 1.000000 1.000000 1.000000 8662.482500  0.000000     0.000000
    11 1.000000 1.000000 1.000000 1.000000 8662.482500  0.000000     0.000000
    12 1.000000 1.000000 1.000000 1.000000 8662.482500  0.000000     0.000000
     9 0.999998 0.999998 0.999998 0.999998 8662.327503  0.000004    -0.154997
     8 0.999990 0.999990 0.999990 0.999990 8661.707538  0.000020    -0.774962
     7 0.995000 0.995000 0.995000 0.995000 8282.121445  0.010000  -380.361055
     5 0.306961 1.000000 1.000000 1.000000 4476.805211  0.693039 -4185.677289
     4 0.310812 0.981722 0.896632 0.984505 2945.270011  0.697309 -5717.212489
     3 0.306961 0.992556 0.914739 0.990622 3290.429271  0.698367 -5372.053229


In [4]:
# =============================================================================
# SURROGATE MODELS: RAW-y GP AND log1p(y) GP
# =============================================================================
# Both surrogates are ARD Matérn-5/2 GPs. Leave-one-out cross-validation scores
# are computed on the original yield scale. The log1p model handles the very wide
# range of yields (about 0.1 to 8,700) and is the more accurate of the two.

# Raw-y GP
raw_rmse, raw_mae, raw_r2 = scores(y, loo_gp(X, y))
gp_f5 = gp_model(X.shape[1]).fit(X, y)
print(f'RAW GP LOOCV: RMSE={raw_rmse:.6f}, MAE={raw_mae:.6f}, R²={raw_r2:.6f}')
print('Kernel:', gp_f5.kernel_)
print(ard_table(gp_f5).to_string(index=False))

# log1p(y) GP
y_log = np.log1p(y)
log_rmse, log_mae, log_r2 = scores(y, np.expm1(loo_gp(X, y_log, log_target=True)))
gp_f5_log = gp_model(X.shape[1], log_target=True).fit(X, y_log)
print(f'\nLOG GP LOOCV: RMSE={log_rmse:.6f}, MAE={log_mae:.6f}, R²={log_r2:.6f}')
print('Kernel:', gp_f5_log.kernel_)
print(ard_table(gp_f5_log).to_string(index=False))

RAW GP LOOCV: RMSE=512.571119, MAE=348.035867, R²=0.977904
Kernel: 0.874**2 * Matern(length_scale=[0.139, 100, 0.59, 100], nu=2.5) + WhiteKernel(noise_level=1e-06)
Feature  Length_Scale  Sensitivity_1_over_L
     x1      0.138613              7.214311
     x3      0.590374              1.693842
     x2    100.000000              0.010000
     x4    100.000000              0.010000

LOG GP LOOCV: RMSE=44.829073, MAE=31.598319, R²=0.999831
Kernel: 1.86**2 * Matern(length_scale=[2.93, 0.692, 1.61, 0.826], nu=2.5) + WhiteKernel(noise_level=1e-06)
Feature  Length_Scale  Sensitivity_1_over_L
     x2      0.691546              1.446036
     x4      0.826085              1.210529
     x3      1.606872              0.622327
     x1      2.933669              0.340870


In [5]:
# =============================================================================
# CONDITIONAL LOCAL SEARCH AND RECOMMENDED CANDIDATE
# =============================================================================
# Searches the high-yield corner of the domain on a dense grid over x2, x3, x4
# with x1 held fixed at the incumbent. The raw-y GP is scored with posterior
# mean, EI, PI and UCB; the log1p GP is scored with its posterior mean on the
# same grid as a robustness check.

fixed_idx = 0
free_idx = [j for j in range(X.shape[1]) if j != fixed_idx]
local_bounds = {feature_cols[j]: (max(DOMAIN[0], DOMAIN[1] - width), DOMAIN[1])
                for j, width in zip(free_idx, LOCAL_FRACTIONS)}

axes = [np.linspace(*local_bounds[feature_cols[j]], GRID_RES) for j in free_idx]
mesh = np.meshgrid(*axes, indexing='ij')
X_local = np.tile(incumbent_x, (mesh[0].size, 1))
for j, values in zip(free_idx, mesh):
    X_local[:, j] = values.ravel()

# Acquisition functions under the raw-y GP
mu, sigma = posterior_search(gp_f5, X_local)
sigma_safe = np.maximum(sigma, 1e-12)
improvement = mu - incumbent_y - XI
z = improvement / sigma_safe
EI = improvement * norm.cdf(z) + sigma_safe * norm.pdf(z)
PI = norm.cdf(z)
UCB = mu + KAPPA * sigma

# Best grid point under each criterion (flagging points on the search boundary)
indices = {name: int(np.argmax(values)) for name, values in
           {'Posterior Mean': mu, 'EI': EI, 'PI': PI, 'UCB': UCB}.items()}
rows = []
for method, idx in indices.items():
    on_boundary = [feature_cols[j] for j in free_idx
                   if np.isclose(X_local[idx, j], local_bounds[feature_cols[j]][0])
                   or np.isclose(X_local[idx, j], local_bounds[feature_cols[j]][1])]
    rows.append({'Method': method, **dict(zip(feature_cols, X_local[idx])),
                 'pred_y': mu[idx], 'pred_std': sigma[idx], 'EI': EI[idx],
                 'Distance_to_Incumbent': np.linalg.norm(X_local[idx] - incumbent_x),
                 'On_boundary': ', '.join(on_boundary)})
print('LOCAL SEARCH — RAW-y GP')
print('Fixed:', feature_cols[fixed_idx], '=', incumbent_x[fixed_idx], '| bounds:', local_bounds)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.6f}'))

# Robustness check: the same grid under the log1p GP
mu_log, std_log = posterior_search(gp_f5_log, X_local, log_target=True)
best_log = int(np.argmax(mu_log))
print('\nLOCAL SEARCH — LOG1P GP (posterior mean)')
print(dict(zip(feature_cols, X_local[best_log])),
      f'predicted y: {mu_log[best_log]:.6f}, approx std: {std_log[best_log]:.6f}')

# Boundary check: the upper corner against the observations nearest to it
upper_corner = np.full(X.shape[1], DOMAIN[1])
unique_idx = np.unique(X, axis=0, return_index=True)[1]
nearest = unique_idx[np.argsort(np.linalg.norm(X[unique_idx] - upper_corner, axis=1))][:5]
pred_y, pred_std = posterior_search(gp_f5_log, X[nearest], log_target=True)
boundary = pd.DataFrame(X[nearest], columns=feature_cols)
boundary['Observed_y'], boundary['Log_GP_pred_y'], boundary['Approx_std'] = y[nearest], pred_y, pred_std
boundary['Distance_to_corner'] = np.linalg.norm(X[nearest] - upper_corner, axis=1)
print('\nOBSERVATIONS NEAREST THE UPPER CORNER')
print(boundary.to_string(index=False, float_format=lambda v: f'{v:.6f}'))

# Recommended query: the raw-y posterior-mean point (the log1p GP agrees)
x_next5 = X_local[indices['Posterior Mean']]
print('\nRaw and log1p GP agree:', bool(np.allclose(x_next5, X_local[best_log])))
print('Recommended next query (posterior mean):')
print(format_point(x_next5))

LOCAL SEARCH — RAW-y GP
Fixed: x1 = 1.0 | bounds: {'x2': (0.9, 1.0), 'x3': (0.85, 1.0), 'x4': (0.9, 1.0)}
        Method       x1       x2       x3       x4      pred_y   pred_std        EI  Distance_to_Incumbent On_boundary
Posterior Mean 1.000000 1.000000 1.000000 1.000000 8662.441967   3.724770  1.460838               0.000000  x2, x3, x4
            EI 1.000000 1.000000 0.850000 1.000000 7738.269372 907.850856 73.075982               0.150000  x2, x3, x4
            PI 1.000000 1.000000 1.000000 0.900000 8662.422818   5.387073  2.114470               0.100000  x2, x3, x4
           UCB 1.000000 1.000000 0.850000 1.000000 7738.269372 907.850856 73.075982               0.150000  x2, x3, x4

LOCAL SEARCH — LOG1P GP (posterior mean)
{'x1': np.float64(1.0), 'x2': np.float64(1.0), 'x3': np.float64(1.0), 'x4': np.float64(1.0)} predicted y: 8662.696034, approx std: 23.966879

OBSERVATIONS NEAREST THE UPPER CORNER
      x1       x2       x3       x4  Observed_y  Log_GP_pred_y  Approx_std  D